
# Module 1: PoC 環境払出とデータ準備（35分）

## 目的
- 数コマンドでの環境払出（Catalog / Schema / Volume）
- サンプルデータの生成と Delta テーブル化
- 「簡易申請書 → 即時払出」のアナロジーを体験

## FE 制約
- 外部ストレージ接続不可 → マネージド Volume/テーブルで代替
- DBFS 不可 → UC Volume を使用


## Step 1: ハンズオン用カタログ・スキーマの決定

Free Edition ではワークスペースに既定のカタログが割り当てられています。
そのカタログ内にハンズオン用のスキーマを作成します。

> **注意**: FE では新規カタログの作成が制限される場合があります。
その場合は `current_catalog()` で返る既定カタログをそのまま使ってください。

In [0]:
%run ./config

In [0]:
# config (%run ./config) から自動設定済み
print(f"現在の既定カタログ: {CATALOG}")
print(f"ユーザー固有スキーマ  : {USER_SCHEMA}")

In [0]:
# config (%run ./config) から自動設定済み
# CATALOG / SCHEMA / USER_SCHEMA / VOLUME_NAME は全ノートブック共通
# テーブル書き込みは USER_SCHEMA に分離→複数受講者が同時実行しても競合しない

print(f"CATALOG     : {CATALOG}")
print(f"USER_SCHEMA : {USER_SCHEMA}  ← テーブルはここに書き込みます")
print(f"VOLUME      : /Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}")


## Step 2: Catalog / Schema / Volume の作成（= 環境払出）

従来型: 申請書 → 承認 → インフラ構築 → 数週間

**Databricks**: 以下の SQL で即時払出↓

In [0]:
# FE では CREATE CATALOG が制限される場合があるため、既定カタログ内に Schema と Volume を作成
try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
    print(f"✅ Catalog 確認/作成: {CATALOG}")
except Exception as e:
    # FE では既定カタログが既に存在するのでエラーでも続行可能
    print(f"ℹ️ Catalog は既存または作成権限なし（既定カタログを使用）: {e}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME_NAME}")

print(f"✅ 環境払出完了: {CATALOG}.{SCHEMA}")
print(f"   Volume: /Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}")
print()
print(f"📌 他のノートブックでは以下を先頭に設定してください:")
print(f'   CATALOG = "{CATALOG}"')
print(f'   SCHEMA = "{SCHEMA}"')


## Step 3: サンプルデータの生成

車両ログを模したデータを `spark.createDataFrame` で生成し、Delta テーブルとして保存します。

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from datetime import datetime, timedelta
import random

# サンプル車両ログデータ
random.seed(42)
vehicles = ["VH-001", "VH-002", "VH-003", "VH-004", "VH-005"]
locations = ["東京", "大阪", "名古屋", "福岡", "札幌"]

rows = []
base_time = datetime(2025, 1, 1)
for i in range(500):
    rows.append((
        f"LOG-{i:04d}",
        random.choice(vehicles),
        random.choice(locations),
        random.uniform(20.0, 95.0),  # battery_pct
        random.randint(0, 150),       # speed_kmh
        random.uniform(10.0, 40.0),   # temperature_c
        base_time + timedelta(hours=random.randint(0, 720))
    ))

schema = StructType([
    StructField("log_id", StringType(), False),
    StructField("vehicle_id", StringType(), False),
    StructField("location", StringType(), False),
    StructField("battery_pct", DoubleType(), False),
    StructField("speed_kmh", IntegerType(), False),
    StructField("temperature_c", DoubleType(), False),
    StructField("timestamp", TimestampType(), False),
])

df = spark.createDataFrame(rows, schema=schema)
df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{USER_SCHEMA}.vehicle_logs")

print(f"✅ Delta テーブル作成完了: {CATALOG}.{USER_SCHEMA}.vehicle_logs ({df.count()} 行)")


## Step 4: データ確認

In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.{USER_SCHEMA}.vehicle_logs LIMIT 10"))


## Step 5: テーブルにコメントを付与（メタデータ駆動）

UC のメタデータは Genie Code やリネージで活用されます。

In [0]:
spark.sql(f"""
  COMMENT ON TABLE {CATALOG}.{USER_SCHEMA}.vehicle_logs IS
  'EV 車両の走行ログ。バッテリー残量、速度、温度、位置情報を含む。'
""")
spark.sql(f"COMMENT ON COLUMN {CATALOG}.{USER_SCHEMA}.vehicle_logs.battery_pct IS 'バッテリー残量 (%)'") 
spark.sql(f"COMMENT ON COLUMN {CATALOG}.{USER_SCHEMA}.vehicle_logs.speed_kmh IS '走行速度 (km/h)'") 
spark.sql(f"COMMENT ON COLUMN {CATALOG}.{USER_SCHEMA}.vehicle_logs.temperature_c IS '外気温 (℃)'") 

print("✅ テーブル/カラムコメント設定完了")


## 補足: 払出の自動化・標準化

実運用では以下を併用し、「申請→即時払出」を自動化できます:

- **Git フォルダ**: ノートブックのバージョン管理
- **Declarative Automation Bundles (DABs)**: テンプレートから同一環境を再現
- **Terraform / Databricks SDK**: IaC でワークスペース構成をコード化

これにより「テンプレから同一環境を再現」= 払出の自動化・標準化が実現します。


## ✅ 完了条件

- [x] 自分の Catalog に Delta テーブルが 1 つできた
- [x] `SELECT * FROM <catalog>.<schema>.vehicle_logs` でデータが返る
- [x] Volume が作成された